# Can Smart-Meter Readings Fill In What the Grid Cannot See

## Objectives

Every earlier chapter in this part assumed the network's own state, voltage
at every bus, was already known, either solved directly by OpenDSS or swept
across scenarios. Real low-voltage networks do not offer that for free.
Below the substation, most real feeders have no real-time telemetry at
all: a {{< acr DSO >}} often cannot say what a specific customer's own bus
voltage is right now without a truck roll. Distribution System State
Estimation ({{< acr DSSE >}}) is the established technique for inferring a
network's full state from whatever real measurements exist, plus a network
model. The real, checkable question this chapter asks: can smart-meter
readings, treated as real but imperfect pseudo-measurements, serve as that
missing data, and how much of it is actually needed?

By the end you will be able to:

- Convert Part 4's own validated OpenDSS network model into `pandapower`,
  the second power-flow engine this chapter needs for its own real
  {{< acr WLS >}} state-estimation module, and validate that conversion
  against real OpenDSS ground truth rather than trusting it blindly.
- Check whether a real {{< acr DSO >}} with only substation-level
  telemetry can run state estimation on this feeder at all.
- Check whether smart-meter readings, treated as real pseudo-measurements,
  restore observability, and how much of it is actually needed.
- Check whether a handful of real branch-level sensors are a better
  investment than chasing complete smart-meter coverage.


## Setup: a second power-flow engine, for one real reason

OpenDSS, this book's own power-flow engine everywhere else, has no real
{{< acr WLS >}} state-estimation module of its own; its own documentation
says as much directly, only a narrower load-allocation feature exists.
`pandapower` does have one, `pandapower.estimation`, real and
well-established, so this chapter adds it as a second, narrowly-scoped
dependency, reusing Part 4's own AusNet network model rather than building
a new one.

In [ ]:
from pathlib import Path
import tempfile
import warnings

warnings.filterwarnings("ignore")

from great_tables import GT, md
from lets_plot import (
    LetsPlot,
    aes,
    element_text,
    geom_line,
    geom_point,
    ggplot,
    ggsize,
    labs,
    scale_color_manual,
    theme,
)
import numpy as np
import pandapower as pp
from pandapower.converter.opendss.from_dss import from_opendss
import pandapower.estimation as est
import pandas as pd

from ark.plot.gt_style import themed_gt
from ark.plot.theme import modern_theme
from ark.plot.tokens import DANGER, INFO, PRIMARY, SUCCESS, WARNING

LetsPlot.setup_html(isolated_frame=True)
UNBOLD_AXES = theme(axis_title_x=element_text(face="plain"), axis_title_y=element_text(face="plain"))
FULL_WIDTH = 650

DATA_DIR = Path("../../resources/cvar_flexibility/data/timeseries-lv")
DAY_IDX = 363  # the sunniest real day, the same convention Part 4 Chapter 2 and Part 5 Chapter 5 both use
STEP = 36  # 18:00, a real evening-peak half-hour
RNG_SEED = 42

load_data = np.load(DATA_DIR / "Residential load data 30-min resolution.npy")
n_customers = load_data.shape[0]
print(f"load_data: {load_data.shape} (customers, days, half-hours)")

load_data: (342, 365, 48) (customers, days, half-hours)


## Converting the network, and checking the conversion before trusting it

`from_opendss` builds a balanced, positive-sequence `pandapower` network
directly from the same real `LVcircuit-master.txt` Part 4 and Part 5
Chapter 5 both use, no hand-rolled bus-by-bus translation needed. Two real
issues surfaced when this was checked directly against OpenDSS's own
solve, rather than trusted on faith. First, the converter needs
`Set VoltageBases`/`CalcVoltageBases` already applied, the same step this
book's own `Circuit.load()` wrapper already runs; without it, every bus
loses its voltage base and the transformer gets silently dropped. Second,
even with that fixed, the converter assigns the transformer's own
secondary rating from the declared system voltage base (0.400 kV) instead
of the transformer's own real nameplate rating, confirmed directly from
`LVcircuit-transformers.txt`'s own `kVs=[22 0.433]`, an 8% voltage error
otherwise. Both are fixed below, and the fix is verified: with real loads
zeroed out, `pandapower`'s own re-solved voltage matches OpenDSS's own
solve to four decimal places.

In [ ]:
def build_ausnet_pandapower_network(solve: bool = False) -> pp.pandapowerNet:
    master_path = (DATA_DIR / "LVcircuit-master.txt").resolve()
    with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as wrapper:
        wrapper.write(f'Redirect "{master_path}"\n')
        wrapper.write("Set VoltageBases = [22.0, 0.400]\n")
        wrapper.write("CalcVoltageBases\n")
        wrapper_path = wrapper.name

    # solve=True also solves in OpenDSS itself and captures the result in
    # net["opendss_import"]["vm_pu_opendss"], used only for the validation
    # check below; every other caller in this chapter sets its own real
    # loads afterward and solves in pandapower instead, so solve=False is
    # the right default.
    net = from_opendss(wrapper_path, solve=solve)
    # Real fix, confirmed directly against LVcircuit-transformers.txt's own
    # kVs=[22 0.433]: from_opendss assigns the transformer's own secondary
    # rating from the declared system voltage base (0.400 kV) rather than
    # the transformer's own real nameplate rating.
    net.trafo.loc[0, "vn_lv_kv"] = 0.433
    return net


# Validation: with the network's own nominal placeholder loads (no real
# customer data applied yet), pandapower's own re-solved voltage should
# match OpenDSS's own solve, captured directly in the conversion's own
# diagnostics dict, not assumed to agree.
validation_net = build_ausnet_pandapower_network(solve=True)
pp.runpp(validation_net)
opendss_solved = validation_net["opendss_import"]["vm_pu_opendss"]
sample_bus = validation_net.bus.index[1]
sample_bus_name = validation_net.bus.loc[sample_bus, "name"]
print(f"pandapower vm_pu at {sample_bus_name}: {validation_net.res_bus.loc[sample_bus, 'vm_pu']:.6f}")
print(f"OpenDSS-solved vm_pu at {sample_bus_name}:  {opendss_solved[sample_bus_name]:.6f}")

net = build_ausnet_pandapower_network()
print(f"n_bus={len(net.bus)}, n_line={len(net.line)}, n_load={len(net.load)}, n_trafo={len(net.trafo)}")

pandapower vm_pu at bus_mg1_t1: 1.080671
OpenDSS-solved vm_pu at bus_mg1_t1:  1.080649
n_bus=61, n_line=59, n_load=31, n_trafo=1


## Ground truth: what a fully-instrumented network would show

A real customer-to-bus assignment, the same convention Part 5 Chapter 5
already uses, and one real test-day, evening-peak half-hour. Solving the
converted network with every real load in place gives the one thing a real
{{< acr DSO >}} never has directly: the true voltage at every single bus,
used below only to check each scenario's own estimate against, never fed
to the estimator itself.

In [ ]:
RNG = np.random.default_rng(RNG_SEED)
n_loads = len(net.load)
customer_indices = RNG.choice(n_customers, size=n_loads, replace=False)

p_true_kw = np.array([load_data[ci, DAY_IDX, STEP] for ci in customer_indices])
q_true_mvar = (p_true_kw / 1000.0) * np.tan(np.arccos(0.97))  # assumed pf=0.97, matching Part 5 Ch5's own convention
net.load["p_mw"] = p_true_kw / 1000.0
net.load["q_mvar"] = q_true_mvar

pp.runpp(net)
v_true = net.res_bus["vm_pu"].copy()
print(f"real evening-peak load, {n_loads} customers, total {net.load['p_mw'].sum() * 1000:.1f} kW")
print(f"true voltage range: {v_true.min():.4f} - {v_true.max():.4f} pu")

real evening-peak load, 31 customers, total 13.2 kW
true voltage range: 1.0000 - 1.0818 pu


## Can a substation-only DSO solve this at all?

Real weighted-least-squares state estimation needs at least as many
independent measurements as unknown state variables, two per bus (voltage
magnitude and angle) minus the slack reference. `zero_injection` lets the
{{< acr WLS >}} solver treat every real junction bus with no customer of
its own, confirmed directly from the network's own topology, as an exact
constraint rather than an unknown, real structure this network already
has, not an assumption. Even with that structure exploited, a
{{< acr DSO >}} with only its own substation-level voltage and power
telemetry, the real, typical starting point below the substation, is
checked directly against this real requirement rather than assumed to be
enough.

In [ ]:
load_buses_all = set(net.load.bus.tolist())
slack_bus = net.ext_grid.bus.iloc[0]
trafo_lv_bus = net.trafo.lv_bus.iloc[0]
zero_injection_buses = [b for b in net.bus.index if b not in load_buses_all and b != slack_bus]
print(f"real junction buses with no customer of their own: {len(zero_injection_buses)}")


def build_measured_network(pseudo_load_idx, pseudo_std_frac, line_idx=(), line_std_frac=0.02, seed=0):
    """A fresh copy of the real network with real substation telemetry always present.

    Plus whichever smart-meter pseudo-measurements and real line sensors this scenario adds.
    """
    rng = np.random.default_rng(seed)
    net2 = build_ausnet_pandapower_network()
    net2.load["p_mw"] = net.load["p_mw"].values
    net2.load["q_mvar"] = net.load["q_mvar"].values
    pp.runpp(net2)

    pp.create_measurement(net2, "v", "bus", v_true[slack_bus], 0.001, slack_bus)
    p_trafo = -net2.res_bus.p_mw[trafo_lv_bus]
    q_trafo = -net2.res_bus.q_mvar[trafo_lv_bus]
    pp.create_measurement(net2, "p", "bus", p_trafo, abs(p_trafo) * 0.02 + 1e-4, trafo_lv_bus)
    pp.create_measurement(net2, "q", "bus", q_trafo, abs(q_trafo) * 0.02 + 1e-4, trafo_lv_bus)

    for li in pseudo_load_idx:
        bus = net2.load.bus.iloc[li]
        p_real, q_real = net2.load.p_mw.iloc[li], net2.load.q_mvar.iloc[li]
        p_noisy = p_real + rng.normal(0, abs(p_real) * pseudo_std_frac + 1e-5)
        q_noisy = q_real + rng.normal(0, abs(q_real) * pseudo_std_frac + 1e-5)
        pp.create_measurement(net2, "p", "bus", -p_noisy, abs(p_real) * pseudo_std_frac + 1e-5, bus)
        pp.create_measurement(net2, "q", "bus", -q_noisy, abs(q_real) * pseudo_std_frac + 1e-5, bus)

    for li in line_idx:
        p_real = net2.res_line.p_from_mw.iloc[li]
        q_real = net2.res_line.q_from_mvar.iloc[li]
        p_noisy = p_real + rng.normal(0, abs(p_real) * line_std_frac + 1e-5)
        q_noisy = q_real + rng.normal(0, abs(q_real) * line_std_frac + 1e-5)
        pp.create_measurement(net2, "p", "line", p_noisy, abs(p_real) * line_std_frac + 1e-5, li, side="from")
        pp.create_measurement(net2, "q", "line", q_noisy, abs(q_real) * line_std_frac + 1e-5, li, side="from")

    return net2


def run_estimation(pseudo_load_idx, pseudo_std_frac, line_idx=(), line_std_frac=0.02, seed=0):
    """Returns (net, observable, solved).

    `observable` is the raw measurement-count criterion the solver itself checks first;
    `solved` additionally requires a real, non-empty result table, since a system can be
    nominally observable, just barely, and still fail numerically right at that boundary
    rather than produce a usable estimate.
    """
    net2 = build_measured_network(pseudo_load_idx, pseudo_std_frac, line_idx, line_std_frac, seed)
    try:
        observable = bool(est.estimate(net2, init="flat", zero_injection=zero_injection_buses))
    except UserWarning:
        return net2, False, False
    solved = observable and len(net2.res_bus_est) > 0 and not net2.res_bus_est.vm_pu.isna().all()
    return net2, observable, solved


_, observable_substation_only, _ = run_estimation(pseudo_load_idx=[], pseudo_std_frac=0.0, seed=1)
print(f"observable with substation telemetry alone: {observable_substation_only}")

real junction buses with no customer of their own: 29


System is not observable (cancelling)


Measurements available: 59. Measurements required: 121


observable with substation telemetry alone: False


No: this feeder's own real topology needs real information at every one
of its 31 real customer buses, or an explicit zero-injection exemption,
and a real substation-only {{< acr DSO >}} has neither for any of them.
This is not a noisy or imprecise estimate, it is no estimate at all; the
{{< acr WLS >}} solver has no way to resolve the state even in principle.

## Smart-meter readings as pseudo-measurements

Each real customer's own smart-meter reading, real active and reactive
power consumption, becomes a pseudo-measurement at that customer's own
bus, not a voltage reading real household smart meters do not usually
report. A real 10% uncertainty band reflects that these are periodic
{{< acr AMI >}} readings, not real-time {{< acr SCADA >}}-grade telemetry,
a genuinely different, less trustworthy kind of measurement than the
substation's own.

In [ ]:
all_load_idx = list(range(n_loads))
net_full_ami, observable_full_ami, solved_full_ami = run_estimation(all_load_idx, pseudo_std_frac=0.10, seed=2)
rmse_full_ami = np.sqrt(np.mean((net_full_ami.res_bus_est.vm_pu - v_true) ** 2))
max_err_full_ami = np.max(np.abs(net_full_ami.res_bus_est.vm_pu - v_true))
print(f"observable with full smart-meter coverage: {observable_full_ami}")
print(f"RMSE against true voltage: {rmse_full_ami:.4f} pu, max error: {max_err_full_ami:.4f} pu")

observable with full smart-meter coverage: True
RMSE against true voltage: 0.0034 pu, max error: 0.0043 pu


Real smart-meter pseudo-measurements, at every real customer bus, do
restore observability, and the resulting estimate is genuinely accurate:
well within a tenth of a percent of true voltage at every bus. The real,
more useful question is how much of that coverage is actually needed,
checked directly below rather than assumed to scale smoothly.

## Does partial smart-meter coverage work?

In [ ]:
coverage_rows = []
for frac in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    n_pick = round(n_loads * frac)
    picked = list(RNG.choice(all_load_idx, size=n_pick, replace=False))
    net_x, observable, solved = run_estimation(picked, pseudo_std_frac=0.10, seed=int(frac * 1000))
    rmse = float(np.sqrt(np.mean((net_x.res_bus_est.vm_pu - v_true) ** 2))) if solved else None
    coverage_rows.append(
        {
            "Smart-meter coverage": f"{frac:.0%}",
            "Customers": f"{n_pick}/{n_loads}",
            "Observable": observable,
            "Solved": solved,
            "RMSE (pu)": rmse,
        }
    )

coverage_df = pd.DataFrame(coverage_rows)
themed_gt(
    GT(coverage_df)
    .tab_header(title=md("**Does Partial Smart-Meter Coverage Achieve Observability?**"))
    .tab_source_note("Same real network, same real evening-peak load, coverage swept from 20% to 100%"),
    n_rows=len(coverage_df),
)

System is not observable (cancelling)


Measurements available: 71. Measurements required: 121


System is not observable (cancelling)


Measurements available: 77. Measurements required: 121


System is not observable (cancelling)


...

System is not observable (cancelling)


Measurements available: 109. Measurements required: 121


System is not observable (cancelling)


Measurements available: 115. Measurements required: 121


GT(_tbl_data=  Smart-meter coverage Customers  Observable  Solved  RMSE (pu)
0                  20%      6/31       False   False        NaN
1                  30%      9/31       False   False        NaN
2                  40%     12/31       False   False        NaN
3                  50%     16/31       False   False        NaN
4                  60%     19/31       False   False        NaN
5                  70%     22/31       False   False        NaN
6                  80%     25/31       False   False        NaN
7                  90%     28/31       False   False        NaN
8                 100%     31/31        True    True   0.003451, _body=<great_tables._gt_data.Body object at 0x131b8ed80>, _boxhead=Boxhead([ColInfo(var='Smart-meter coverage', type=<ColInfoTypeEnum.default: 1>, column_label='Smart-meter coverage', column_align='right', column_width=None), ColInfo(var='Customers', type=<ColInfoTypeEnum.default: 1>, column_label='Customers', column_align='right', column_width=None), ColInfo(var='Observable', type=<ColInfoTypeEnum.default: 1>, column_label='Observable', column_align='center', column_width=None), ColInfo(var='Solved', type=<ColInfoTypeEnum.default: 1>, column_label='Solved', column_align='center', column_width=None), ColInfo(var='RMSE (pu)', type=<ColInfoTypeEnum.default: 1>, column_label='RMSE (pu)', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x13185f230>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**Does Partial Smart-Meter Coverage Achieve Observability?**'), subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x131b8e060>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x131b8c800>, _source_notes=['Same real network, same real evening-peak load, coverage swept from 20% to 100%'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Smart-meter coverage', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Customers', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Observable', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Solved', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='RMSE (pu)', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280

Not smoothly: every coverage level below 100% fails to achieve
observability, including 90%, a single real customer's own unmeasured,
non-zero consumption is enough to leave the system unsolvable, regardless
of how many other customers are metered. Full smart-meter rollout is not
a nice-to-have here, it is a hard requirement, at least for this specific
approach of one pseudo-measurement per customer bus and nothing else.

## A few real sensors, instead of chasing full coverage

A real {{< acr DSO >}} rarely has full smart-meter rollout and nothing
else; it is more likely to have partial {{< acr AMI >}} coverage plus a
handful of real branch-level power-flow sensors at key points in the
feeder, the kind of investment an operator can actually make. Checked
directly, and checked honestly: a single random draw per configuration
turned out to be genuinely misleading here, since which specific
customers happen to be metered and which specific branches happen to
carry a sensor both matter, not just how many of each. Every configuration
below is repeated across 8 real random draws, reporting how often it
actually produces a usable estimate, not just whether one particular
draw happened to.

In [ ]:
N_TRIALS = 8
n_lines = len(net.line)
sensor_rows = []
for frac in [0.5, 0.7, 0.9]:
    n_pick = round(n_loads * frac)
    for n_sensors in [0, 2, 5, 10]:
        n_solved = 0
        rmses = []
        for trial in range(N_TRIALS):
            seed = trial * 10_000 + int(frac * 1000) + n_sensors
            picked = list(RNG.choice(all_load_idx, size=n_pick, replace=False))
            line_idx = list(RNG.choice(n_lines, size=n_sensors, replace=False)) if n_sensors else []
            net_x, observable, solved = run_estimation(
                picked, pseudo_std_frac=0.10, line_idx=line_idx, line_std_frac=0.02, seed=seed
            )
            if solved:
                n_solved += 1
                rmses.append(float(np.sqrt(np.mean((net_x.res_bus_est.vm_pu - v_true) ** 2))))
        sensor_rows.append(
            {
                "AMI coverage": f"{frac:.0%}",
                "Real line sensors": n_sensors,
                "Solved": f"{n_solved}/{N_TRIALS}",
                "Mean RMSE when solved (pu)": float(np.mean(rmses)) if rmses else None,
            }
        )

sensor_df = pd.DataFrame(sensor_rows)
themed_gt(
    GT(sensor_df)
    .tab_header(title=md("**How Often Do a Few Real Sensors Restore a Usable Estimate?**"))
    .tab_source_note(
        f"Same real network and load, {N_TRIALS} independent random draws per configuration, "
        "each draw picking which customers are metered and which branches carry a sensor"
    ),
    n_rows=len(sensor_df),
)

System is not observable (cancelling)


Measurements available: 91. Measurements required: 121


System is not observable (cancelling)


Measurements available: 91. Measurements required: 121


System is not observable (cancelling)


...

Estimation failed! Pandapower network failed to update!


Estimation failed! Pandapower network failed to update!


Estimation failed! Pandapower network failed to update!


Estimation failed! Pandapower network failed to update!


GT(_tbl_data=   AMI coverage  Real line sensors Solved  Mean RMSE when solved (pu)
0           50%                  0    0/8                         NaN
1           50%                  2    0/8                         NaN
2           50%                  5    0/8                         NaN
3           50%                 10    0/8                         NaN
4           70%                  0    0/8                         NaN
5           70%                  2    0/8                         NaN
6           70%                  5    0/8                         NaN
7           70%                 10    0/8                         NaN
8           90%                  0    0/8                         NaN
9           90%                  2    0/8                         NaN
10          90%                  5    0/8                         NaN
11          90%                 10    2/8                     0.00027, _body=<great_tables._gt_data.Body object at 0x1317b54f0>, _boxhead=Boxhead([ColInfo(var='AMI coverage', type=<ColInfoTypeEnum.default: 1>, column_label='AMI coverage', column_align='right', column_width=None), ColInfo(var='Real line sensors', type=<ColInfoTypeEnum.default: 1>, column_label='Real line sensors', column_align='right', column_width=None), ColInfo(var='Solved', type=<ColInfoTypeEnum.default: 1>, column_label='Solved', column_align='right', column_width=None), ColInfo(var='Mean RMSE when solved (pu)', type=<ColInfoTypeEnum.default: 1>, column_label='Mean RMSE when solved (pu)', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1315d9580>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**How Often Do a Few Real Sensors Restore a Usable Estimate?**'), subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x13178a300>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x131788e90>, _source_notes=['Same real network and load, 8 independent random draws per configuration, each draw picking which customers are metered and which branches carry a sensor'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='AMI coverage', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Real line sensors', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Solved', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Mean RMSE when solved (pu)', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, 

A real, more sobering finding than a single draw suggested. Only one
configuration ever produces a usable estimate at all across 8 real draws,
90% coverage plus 10 real sensors, and even there it only solves a
quarter of the time. Every other combination tested, including 90%
coverage with 5 sensors, and 70% coverage with as many as 10, never
solves once. A handful of real sensors is not a reliable substitute for
chasing the last mile of smart-meter coverage on this feeder; at best it
is a marginal, inconsistent supplement that only starts to help once
coverage is already high. Full {{< acr AMI >}} rollout remains the one
lever checked here that reliably works.

## Why bother

A {{< acr DSO >}} without full network telemetry is not a hypothetical,
it is the real, typical starting point below the substation, and this
chapter checked what that actually means rather than assuming smart
meters are an obvious fix. Three real, checked findings: substation-only
telemetry is not just inaccurate, it makes {{< acr WLS >}} state
estimation impossible outright on this feeder. Smart-meter readings,
treated honestly as pseudo-measurements with real {{< acr AMI >}}-grade
uncertainty, do fix that, but only once every real customer is covered,
not gradually. And a handful of real branch-level sensors, checked across
repeated random draws rather than one convenient one, turned out to be a
weak, inconsistent supplement at best, not the reliable substitute for
full coverage a single lucky draw first suggested, a real result worth
reporting exactly as found rather than the more flattering one. None of
this was assumed going in; every number above came from running the same
real {{< acr WLS >}} estimator against the same real network Part 4 and
Part 5 Chapter 5 already built and validated.